# End_to_End_Pipeline - Pipeline Orchestrator


### Load Shared Configuration and Helper Functions
Loads config values, `logger`, and the `validate_row_count()` helper from `00_Config_Utils`.

In [0]:
%run ./00_Config_Utils

# logging - Shared Configuration and Helper Functions

This keeps the project **modular** (reusable functions instead of copy-pasted code) and **parameterized** (thresholds and table names live in one place, not hardcoded everywhere).

We use Python's built-in `logging` module instead of plain `print()` statements. This gives every message a timestamp and a severity level (INFO, WARNING, ERROR), which is standard practice in real data pipelines.

Every layer (Bronze, Silver, Gold) saves DataFrames as Delta tables the same way. This function avoids repeating that logic in every notebook.

### Step 1: Set Up Logging


2026-07-11 20:15:41,464 - INFO - Logger initialized for ServiceTrack pipeline.


### Step 2: Configuration Values (Parameterized Settings)


2026-07-11 20:15:41,686 - INFO - Received command c on object id p0
2026-07-11 20:15:41,730 - INFO - Configuration values loaded.


### Step 3: Reusable Function - Load a CSV File


### Step 4: Reusable Function - Save a DataFrame as a Delta Table


### Step 5: Reusable Function - Convert Multiple Columns to Date Type


### Step 6: Reusable Function - Validate Row Counts


In [0]:
# 'col' is already imported via 00_Config_Utils, so no separate import is needed here.

###  Run the Bronze Layer Notebook

In [0]:
# Run the Bronze Layer notebook, with error handling so a failure here stops the pipeline with a clear message
try:
    dbutils.notebook.run("./02_Bronze_Layer", 0)
    logger.info("Bronze Layer notebook completed.")
except Exception as e:
    logger.error(f"Bronze Layer notebook failed: {e}")
    raise

2026-07-11 20:16:24,816 - INFO - Bronze Layer notebook completed.


### Run the Silver Layer Notebook

In [0]:
# Run the Silver Layer notebook
try:
    dbutils.notebook.run("./03_Silver_Layer", 0)
    logger.info("Silver Layer notebook completed.")
except Exception as e:
    logger.error(f"Silver Layer notebook failed: {e}")
    raise

2026-07-11 20:16:25,007 - INFO - Received command c on object id p0
2026-07-11 20:16:56,425 - INFO - Silver Layer notebook completed.


### Run the Gold Layer Notebook


In [0]:
# Run the Gold Layer notebook
try:
    dbutils.notebook.run("./04_Gold_Layer", 0)
    logger.info("Gold Layer notebook completed.")
except Exception as e:
    logger.error(f"Gold Layer notebook failed: {e}")
    raise

2026-07-11 20:16:56,520 - INFO - Received command c on object id p0
2026-07-11 20:17:38,044 - INFO - Gold Layer notebook completed.


### Validate Row Counts Across Layers


In [0]:
# Load Delta tables to validate, using config table names
bronze_count = spark.read.table(BRONZE_SERVICE_JOBS_TABLE).count()
silver_count = spark.read.table(SILVER_ENRICHED_JOBS_TABLE).count()

# Use the shared validate_row_count() helper instead of a manual if/else block
validate_row_count(bronze_count, EXPECTED_BRONZE_JOB_COUNT, "Bronze service jobs")
validate_row_count(silver_count, EXPECTED_SILVER_JOB_COUNT, "Silver enriched jobs")

2026-07-11 20:17:39,898 - INFO - Success: Bronze service jobs row count matched expected value (1510).
2026-07-11 20:17:39,899 - INFO - Success: Silver enriched jobs row count matched expected value (1500).


True

### Validate: technician_name Has No Nulls
This confirms the blank technician_name values (originally ~75 rows) were successfully filled during the Silver layer step.

In [0]:
# Check that technician_name has zero nulls in the Silver table
null_tech_count = spark.read.table(SILVER_ENRICHED_JOBS_TABLE).filter(col("technician_name").isNull()).count()

if null_tech_count == 0:
    logger.info("Success: No null technician_name values found.")
else:
    logger.warning(f"Warning: {null_tech_count} technician_name values are still null. Review the Silver layer lookup logic.")

2026-07-11 20:17:40,603 - INFO - Success: No null technician_name values found.


### Validate: No Negative Costs

In [0]:
# Check for negative costs
df_silver_check = spark.read.table(SILVER_ENRICHED_JOBS_TABLE)
negative_cost_count = df_silver_check.filter(
    (col("estimated_cost") < 0) | (col("actual_cost") < 0)
).count()

if negative_cost_count == 0:
    logger.info("Success: No negative costs found.")
else:
    logger.warning(f"Warning: {negative_cost_count} rows have negative costs. Review the data quality rules.")

2026-07-11 20:25:14,334 - INFO - Success: No negative costs found.


### Query and Display Sample Outputs 

In [0]:
# Query sample gold results using config table names
logger.info("Displaying sample Technician Performance output:")
display(spark.read.table(GOLD_TECHNICIAN_PERFORMANCE_TABLE).limit(3))

logger.info("Displaying sample Customer Latest Visit output:")
display(spark.read.table(GOLD_CUSTOMER_LATEST_VISIT_TABLE).limit(3))

2026-07-11 20:28:38,496 - INFO - Displaying sample Technician Performance output:


technician_id,technician_name,total_jobs,completed_jobs,avg_repair_duration_days,delayed_jobs,delay_rate_pct
T006,Deepak Sharma,186,135,5.246478873239437,63,46.666666666666664
T001,Rajesh Kumar,182,132,2.9791666666666665,0,0.0
T005,Kavitha Nair,192,143,2.569620253164557,0,0.0


2026-07-11 20:28:39,938 - INFO - Displaying sample Customer Latest Visit output:


customer_id,customer_name,job_id,received_date,job_status,actual_cost
CUST0001,Kiran Patel,JOB00991,2024-03-13,Cancelled,null
CUST0003,Bhavna Tiwari,JOB00107,2024-03-18,Completed,2193.57
CUST0004,Syed Rao,JOB00193,2024-03-16,Pending,null


### Print Overall Pipeline Success Message


In [0]:
logger.info("End-to-end pipeline validation complete.")
print("==================================================")
print("   END-TO-END PIPELINE COMPLETED SUCCESSFULLY!    ")
print("==================================================")

2026-07-11 20:29:14,177 - INFO - End-to-end pipeline validation complete.


   END-TO-END PIPELINE COMPLETED SUCCESSFULLY!    
